In [2]:
pip install sentence-transformers faiss-cpu

   ---------------------------------------- 0.0/10.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.1 MB ? eta -:--:--
   - -------------------------------------- 0.3/10.1 MB ? eta -:--:--
   - -------------------------------------- 0.3/10.1 MB ? eta -:--:--
   - -------------------------------------- 0.3/10.1 MB ? eta -:--:--
   - -------------------------------------- 0.3/10.1 MB ? eta -:--:--
   - -------------------------------------- 0.3/10.1 MB ? eta -:--:--
   - -------------------------------------- 0.3/10.1 MB ? eta -:--:--
   -- ------------------------------------- 0.5/10.1 MB 233.0 kB/s eta 0:00:42
   -- ------------------------------------- 0.5/10.1 MB 233.0 kB/s eta 0:00:42
   --- ------------------------------------ 0.8/10.1 MB 302.4 kB/s eta 0:00:31
   ---- ----------------------------------- 1.0/10.1 MB 331.1 kB/s eta 0:00:28
   ---- ----------------------------------- 1.0/10.1 MB 331.1 kB/s eta 0:00:28
   ---- ----------------------------------- 1


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:



# ================================
# IMPORTS
# ================================
import json
import os
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from tqdm import tqdm


# ================================
# PATHS
# ================================
input_path = "data/processed/chunks.json"
faiss_index_path = "data/vector_store/faiss_index.bin"
metadata_path = "data/vector_store/metadata.json"

os.makedirs("data/vector_store", exist_ok=True)


# ================================
# LOAD DATA
# ================================
if not os.path.exists(input_path):
    raise FileNotFoundError("Run chunking step first!")

with open(input_path, "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"Loaded {len(chunks)} chunks")


# ================================
# LOAD BETTER EMBEDDING MODEL 🔥
# ================================
print("\n🔹 Loading embedding model...")

model = SentenceTransformer("BAAI/bge-small-en-v1.5")


# ================================
# QUERY PREFIX FUNCTION 🔥
# ================================
def add_prefix(text):
    return "Represent this sentence for searching relevant passages: " + text


# ================================
# PREPARE TEXTS
# ================================
texts = [add_prefix(chunk["text"]) for chunk in chunks]


# ================================
# GENERATE EMBEDDINGS
# ================================
print("\n🔹 Generating embeddings...\n")

embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"Embeddings shape: {embeddings.shape}")


# ================================
# NORMALIZE (COSINE SIMILARITY)
# ================================
faiss.normalize_L2(embeddings)


# ================================
# BUILD FAISS INDEX
# ================================
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print(f"\n✅ FAISS index built with {index.ntotal} vectors")


# ================================
# SAVE INDEX
# ================================
faiss.write_index(index, faiss_index_path)


# ================================
# IMPROVED METADATA 🔥
# ================================
metadata = []

for i, chunk in enumerate(chunks):
    metadata.append({
        "chunk_id": chunk["chunk_id"],
        "text": chunk["text"],
        "source": chunk["source"],
        "type": chunk["type"],
        "domain": chunk["domain"],
        "title": chunk["text"][:80]  # NEW (for UI + ranking)
    })

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4)

print(f"\n✅ Metadata saved to: {metadata_path}")


# ================================
# TEST RETRIEVAL (IMPROVED)
# ================================
query = "best fertilizer for wheat crop"

print("\n🔹 Testing retrieval...\n")

query_embedding = model.encode(
    [add_prefix(query)],
    convert_to_numpy=True
)

faiss.normalize_L2(query_embedding)

k = 8  # increased

distances, indices = index.search(query_embedding, k)

print("Query:", query)
print("\nTop Results:\n")

seen = set()

for i, idx in enumerate(indices[0]):
    if idx >= len(metadata):
        continue

    text = metadata[idx]["text"]

    # MMR-style dedup
    if text in seen:
        continue
    seen.add(text)

    print(f"Rank {i+1} | Score: {distances[0][i]:.4f}")
    print(metadata[idx]["title"])
    print("\n----------------------\n")

e:\Udemy ML course\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 148 chunks

🔹 Loading embedding model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1625.82it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🔹 Generating embeddings...



Batches: 100%|██████████| 5/5 [01:02<00:00, 12.59s/it]


Embeddings shape: (148, 384)

✅ FAISS index built with 148 vectors

✅ Metadata saved to: data/vector_store/metadata.json

🔹 Testing retrieval...

Query: best fertilizer for wheat crop

Top Results:

Rank 1 | Score: 0.7599
fertilizer or converted to different other n fertilizers. maintaining sufficient

----------------------

Rank 2 | Score: 0.7453
if new lands are available these are often less productive. the need will probab

----------------------

Rank 3 | Score: 0.7348
and geographies are required in developing the decision rules for nutrient exper

----------------------

Rank 4 | Score: 0.7336
perspectives. the challenge ahead is to manage fertilizers and soil in a sustain

----------------------

Rank 5 | Score: 0.7319
fields condition and vegetation density. for this purpose, eosda crop monitoring

----------------------

Rank 6 | Score: 0.7317
some steps to help you determine the best type of commercial fertilizer conduct 

----------------------

Rank 7 | Score: 0.7304
soil